# 15 — Framework thinking: what they buy, and the archetypes

**What you'll learn**

- Why the machinery you built by hand across Parts 1–4 *already is* a framework — the loop, router, verifier, budget, gate, checkpoint, tracer, context tools, and sub-agents sitting in `shoplab`
- What a framework actually adds on top of that — defaults, integrations, and a community — and what you trade for it: control and transparency
- The **five archetypes** every agent library sorts into — `graph`, `typed`, `team`, `model-driven loop`, `code-as-action` — and which piece of your own code each one repackages
- A checklist that turns a real requirement (human-in-the-loop? durable state? a specific provider?) into an archetype, and points at the Part 5 chapter that builds it
- When *not* to reach for a framework at all, following Anthropic's [Building effective agents](https://www.anthropic.com/engineering/building-effective-agents)

*Time: ~2 min. Cost: ~$0.01 (one small triage run). Cached reruns are free.*

## You have already built a framework

Stop and take inventory. Across four parts you did not just *use* an agent — you built the parts an agent is made of, one file at a time. A tool-calling **loop** (`shoplab.loop.run_agent`, ch02). A **router** that branches on the model's output (ch05). A **verifier** that grades a decision against a rubric (ch06). A **budget** that caps calls and dollars, a **gate** that stops a risky tool for a human, and a **checkpoint** that lets a crashed run resume (`shoplab.controls`, ch08). A **tracer** that records every step as a span (ch03). **Context tools** that compact and offload a bloated message list (ch10). **Sub-agents** a lead delegates to (ch07). And two wire protocols — **MCP** for tools and **A2A** for peers (ch13–14).

That collection has a name. When a library ships the loop, the router, the gate, the checkpointer, and the tracer *for* you — wired together with sensible defaults — we call it a framework. You have been building one all along: an un-branded, fully transparent framework. Part 5 holds yours up against the libraries the field actually reaches for, so the question is never "what is this magic?" but "which of my parts did they package, and how?\"

In [ ]:
# === config (identical in every notebook) ===
import os, getpass
import litellm
from dotenv import load_dotenv              # pip install -e ".[obs]" if this fails

load_dotenv(".env")   # reads OPENROUTER_API_KEY / MODEL / STRONG_MODEL (see .env.example)

if not os.environ.get("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass.getpass("OpenRouter API key: ")

MODEL = os.environ.get("MODEL", "openrouter/deepseek/deepseek-v3.2")
STRONG_MODEL = os.environ.get("STRONG_MODEL", "openrouter/deepseek/deepseek-v4-flash")

# Per-notebook override: uncomment to ignore .env here (any LiteLLM provider works).
# MODEL = "openrouter/google/gemini-2.5-flash-lite"
# MODEL = "openai/gpt-4o-mini"              # direct OpenAI, uses OPENAI_API_KEY instead

TEMPERATURE = 0                             # the whole course runs at temperature 0
litellm.drop_params = True                  # ignore params a provider does not support
litellm.cache = litellm.Cache(type="disk", disk_cache_dir=".litellm_cache")  # reruns are ~free

### Phoenix observability (optional)

In [ ]:
# optional: Phoenix tracing (see notebook 03)
import obs

obs.enable_phoenix()

## The machinery, taking inventory

The thesis is easy to assert and easy to check: every part named above is a real, importable object *right now*. No frameworks are installed in this notebook — the base course stack is all you need, because you wrote the stack. Here is the inventory, pulled live from `shoplab`.

In [ ]:
import shoplab.loop, shoplab.controls, shoplab.trace, shoplab.context, shoplab.verify
from shoplab.tools import standard_tools

built = {
    "loop (ch02)":          shoplab.loop.run_agent,
    "verifier (ch06)":      shoplab.verify.check_decision,
    "budget (ch08)":        shoplab.controls.Budget,
    "gate (ch08)":          shoplab.controls.require_approval,
    "checkpoint (ch08)":    shoplab.controls.Checkpoint,
    "tracer (ch03)":        shoplab.trace.Span,
    "context tools (ch10)": shoplab.context.compact,
    "sub-agents (ch07/10)": shoplab.context.run_subagent,
}
for label, obj in built.items():
    print(f"{label:22} -> {obj.__module__}.{obj.__qualname__}")
print("\nstandard toolset:", sorted(standard_tools()))

> **What you should see:** eight real objects resolve to their `shoplab` modules, plus the nine-tool ops-desk toolset. Nothing here is a stub — this is the same code the last twelve chapters ran. A framework's job is to give you objects like these, already assembled; you happen to own an assembled set already.

## A framework is opinionated packaging

So what does a framework *add*, if you already have the parts? Three things, and only three. **Defaults** — a loop, a checkpointer, and a tool schema that work out of the box, so you write configuration instead of plumbing. **Integrations** — pre-built adapters to model providers, vector stores, tracing backends, and protocols like MCP. And a **community** — examples, issues, and a hiring pool who already know the API.

The bill for those three is paid in the same currency every time: **control and transparency**. When the loop is yours, you can see and change every step; when it is the framework's, you accept its opinion about retries, message trimming, error handling, and where the seams are. That is not a bad trade — it is often the right one — but it is a real trade, and the point of having built the un-framed version is that you can now *price* it. Every chapter in Part 5 is the same exercise: take one archetype, watch a framework package a part you own, and name exactly what you gained and what you gave up.

## The five archetypes

Strip the branding off the dozen agent libraries and they collapse into five shapes. Four of them repackage machinery you already built; the fifth is the one genuinely new execution model in this whole part.

| Archetype | Essence, in one line | The machinery you built | Framework we meet |
|---|---|---|---|
| **graph** | an explicit state machine: nodes do work, edges decide what runs next | the ch02 loop + the ch05 router + the ch08 checkpoint, drawn as a diagram | LangGraph (ch16) |
| **typed** | schema-first IO: declare the input and output types, the library fills the middle | the ch01 model boundary — structured output via `parse_json_loose` | Pydantic AI (appendix) |
| **team** | several roles collaborate; a lead delegates to specialists | the ch07 multi-agent lead + sub-agents (`run_subagent`) | CrewAI (ch18) |
| **model-driven loop** | the model decides every step; the framework just runs the loop | `run_agent` itself — the ch02 tool-calling loop | Strands / OpenAI Agents SDK (appendix) |
| **code-as-action** | the model writes *code* as its action, run in a sandbox | *nothing yet* — a genuinely new execution model | smolagents (ch17) |

Read each row as a claim you can already defend:

- **graph** — [LangGraph](https://docs.langchain.com/oss/python/langgraph/overview) bills itself as a "low-level orchestration framework and runtime for ... long-running, stateful agents." Its nodes-and-edges are your ch05 router made first-class, and its [persistence layer](https://docs.langchain.com/oss/python/langgraph/persistence) is your ch08 checkpoint under a new name.
- **typed** — [Pydantic AI](https://pydantic.dev/docs/ai/overview/) is "a typed, extensible agent loop with every model a string swap away." That is the ch01 model boundary: declare the shape, let the library coerce the model's text into it.
- **team** — [CrewAI](https://docs.crewai.com/) builds "collaborative AI agents, crews, and flows"; the pattern is Anthropic's [orchestrator-workers](https://www.anthropic.com/engineering/multi-agent-research-system) — a lead delegating to specialist subagents, exactly the ch07 shape.
- **model-driven loop** — the [ReAct](https://arxiv.org/abs/2210.03629) reason-act loop, now the default of SDKs like [Strands](https://strandsagents.com/) and the [OpenAI Agents SDK](https://openai.github.io/openai-agents-python/) ("very few abstractions"). It is `run_agent` with a logo.
- **code-as-action** — the one new idea. [CodeAct](https://arxiv.org/abs/2402.01030) shows agents that emit executable Python as their action space beat JSON tool-callers, and [smolagents](https://huggingface.co/docs/smolagents/index) makes that its first-class `CodeAgent`. You have not built this — ch17 does.

## One run, every archetype hiding in it

Enough assertion. Run the ops desk on a real ticket — the same `run_agent` from ch02, the same frozen `MODEL` — and watch the trajectory. Then we label each part of the run with the archetype a framework would use to name it. No new code: the point is that the archetypes are already *in* the loop you have.

In [ ]:
from shoplab.tools import standard_tools, Ledger
from shoplab.loop import run_agent
import shoplab.llm

ledger = Ledger()
desk = standard_tools(ledger)

task = ("Triage this Larkspur return. Order ORD-7312, customer CUST-07, sku LK-1016, qty 1. "
        "Reason: 'I opened the Torrent boots, wore them one evening indoors, they pinch at the "
        "toes. Repacked with tags. Please refund my original payment.' item_condition=opened, "
        "days_since_delivery=18, requested_action=refund, evidence_photo=false. Look up the "
        "order and the customer, search policy for the rule, then call finish with decision, "
        "policy_id, and refund_usd.")
system = ("You are the Larkspur Outfitters ops desk. Gather the order, the customer, and ONE "
          "policy search, then call finish with decision, policy_id, refund_usd. No repeats.")

trajectory = []
result = run_agent(task, desk, system=system, max_steps=8,
                   on_step=lambda step, msg: trajectory.append(
                       [tc.function.name for tc in (msg.tool_calls or [])] or ["<text>"]))
print("trajectory:", trajectory)
print("decision:  ", result.answer, "| stop:", result.stop_reason)
print("cost so far:", round(shoplab.llm.usage_summary()["cost_usd"], 4), "USD")

> **What you should see:** the loop takes a handful of steps — look up the order, look up the customer, search policy, do the arithmetic, then `finish` — and lands on `partial_refund` with a refund near $170.99: the gold decision and amount for TKT-2205 (a 10% restocking fee on an opened return). The policy it cites may be `pol-restocking` (the gold) or the neighbouring `pol-returns`, and the trajectory can wobble (the model may repeat a search or skip `calc`). What is invariant is the decision and the dollar figure — and that *the model chose each step* while the loop merely executed it, for a few tenths of a cent (zero on a cached rerun).

The answer was not free-form. The `finish` tool declares a schema — a decision `enum`, a `policy_id`, an optional `refund_usd` — and the model's final call had to fit it. That declared output shape is the whole of the `typed` archetype, already sitting in your toolset:

In [ ]:
import json
from shoplab.tools import standard_tools

print(json.dumps(standard_tools()["finish"].params, indent=2))

## The same run, labelled by archetype

Nothing above was framework code — yet every archetype is already present. Here is the run, annotated with the name each library would give the piece:

| Part of the run | What it is | Archetype |
|---|---|---|
| the `for step in range(...)` loop in `run_agent` | the model proposes, the loop executes, repeat | **model-driven loop** |
| "if the model called a tool, run it; else finish" | branching on the model's output | **graph** (a conditional edge) |
| the `finish` schema the decision had to fit | a declared output shape | **typed** |
| a lead farming sub-lookups to helpers (ch07) | roles collaborating under an orchestrator | **team** |
| — (this loop calls JSON tools) | the model could instead *write code* to call them | **code-as-action** (ch17) |

Three of the five appear in this exact run: the model-driven loop, the graph's conditional edge, and the typed `finish` schema. The fourth — **team** — is the ch07 shape you already built; it simply did not fire on a ticket one agent could handle alone. The fifth — **code-as-action** — is the one new thing Part 5 teaches.

## How to choose: a checklist

Archetypes are a menu, not a ranking. The honest way to pick is to start from a requirement, read off the archetype — and, usefully, name the exact `shoplab` part you already built that becomes that framework's headline feature.

| Do you need... | Reach for | You built the seed in | Framework version in Part 5 |
|---|---|---|---|
| a human to approve a risky action before it fires | graph **interrupt** | `require_approval` (ch08) | [LangGraph interrupts](https://docs.langchain.com/oss/python/langgraph/interrupts) — ch16 |
| durable state that survives a crash and resumes | graph **checkpointer** | `Checkpoint` (ch08) | [LangGraph persistence](https://docs.langchain.com/oss/python/langgraph/persistence) — ch16 |
| explicit branching you can draw and audit | **graph** | the ch05 router | [LangGraph](https://docs.langchain.com/oss/python/langgraph/overview) — ch16 |
| several specialist roles on one job | **team** | multi-agent (ch07) | [CrewAI](https://docs.crewai.com/) — ch18 |
| the model to write code as its action | **code-as-action** | *(new)* | [smolagents](https://huggingface.co/docs/smolagents/index) — ch17 |
| validated, typed inputs and outputs | **typed** | the ch01 model boundary | [Pydantic AI](https://pydantic.dev/docs/ai/overview/) — appendix |
| to swap model providers freely | **model-driven loop** over LiteLLM | `shoplab.llm.complete` (ch01) | [Strands](https://strandsagents.com/) / [OpenAI Agents SDK](https://openai.github.io/openai-agents-python/) — appendix |
| to reach outside tools or peer agents over a standard wire | **MCP / A2A** | ch13 / ch14 | ch19 |

Notice the third column: there is a piece of your own code behind every framework headline. That is the whole reason we built them first.

## When not to reach for a framework

The most important message in this part is the one that says *maybe don't*. Anthropic's [Building effective agents](https://www.anthropic.com/engineering/building-effective-agents) draws the line first between a **workflow** — "systems where LLMs and tools are orchestrated through predefined code paths" — and an agent that decides its own path, and then argues for reaching for the simplest of these that actually works. A framework is not the simplest thing; it is a bet that its defaults match your problem.

That guidance is worth internalising before ch16. Start with the model API directly — many patterns are a few lines of plain code, as your four parts prove — and add a framework only when its defaults, integrations, or community clearly earn the abstraction. If you do adopt one, understand the machinery underneath it, because that machinery is exactly what you have been building. The cost of a framework is always the same shape: control and transparency traded for defaults and reach. You are now in the rare position of being able to price that trade, because you have run the un-framed version of every part.

## Recap

| Concept | One-liner |
|---|---|
| You built a framework | the loop, router, verifier, budget, gate, checkpoint, tracer, context tools, and sub-agents already live in `shoplab`. |
| Framework = packaging | an opinionated bundle of that machinery, with defaults, integrations, and a community. |
| The trade | you buy defaults and reach; you pay in control and transparency. |
| graph | an explicit state machine — your ch05 router + ch08 checkpoint made first-class (LangGraph, ch16). |
| typed | schema-first IO — your ch01 model boundary (Pydantic AI, appendix). |
| team | roles collaborate under a lead — your ch07 multi-agent (CrewAI, ch18). |
| model-driven loop | the model decides every step — `run_agent` itself (Strands / OpenAI Agents SDK). |
| code-as-action | the model writes code to act — the one new execution model (smolagents, ch17). |
| Choosing | start from a requirement, read off the archetype, find the `shoplab` seed behind it. |
| When not to | reach for the simplest thing that works; a framework must earn its abstraction. |

## Exercises

1. **Map one of your own requirements.** Pick a real feature you would want the Larkspur desk to have — say, "a supervisor signs off before any refund over $200," or "the run must survive a server restart mid-ticket." Which archetype does the checklist send you to, which `shoplab` part is its seed, and which Part 5 chapter builds the framework version?
2. **Find the ch02–12 artifact behind a framework feature you have heard of.** Name one advertised feature of a framework you have seen — LangGraph's "human-in-the-loop," CrewAI's "agents," a "structured output" mode. Trace it to the exact function or class in `shoplab` that does the same job, and write one sentence on what the framework adds beyond your version.
3. **Predict the best archetype for Larkspur triage.** Given what the ops desk actually does — look up facts, apply a policy cascade, sometimes pause for a human, sometimes move money — argue which single archetype fits it best and why. Which parts of the job push you toward `graph`, and which are content with the plain `model-driven loop`? (There is no single right answer; the reasoning is the exercise.)

**Next up:** ch16 — the graph archetype. We rebuild this exact triage as a LangGraph `StateGraph`: your ch05 router becomes a conditional edge, your ch08 checkpoint becomes a `MemorySaver`, and your ch08 approval gate becomes an `interrupt_before` — the same machinery, now packaged.